# Importing Libraries

In [2]:
# ==========================================
# 1. Import Libraries
# ==========================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ==========================================
# 2. Load Dataset
# ==========================================

data = load_breast_cancer()
X = data.data
y = data.target


# ==========================================
# 3. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# ==========================================
# 4. Define Models (Using Pipeline)
# ==========================================

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500))
    ]),

    "Decision Tree": DecisionTreeClassifier(),

    "Random Forest": RandomForestClassifier(n_estimators=200),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}


# ==========================================
# 5. Cross-Validation Comparison
# ==========================================

print("Cross Validation Results (5-Fold):\n")

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_results[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f}")


# ==========================================
# 6. Select Best Model
# ==========================================

best_model_name = max(cv_results, key=cv_results.get)
best_model = models[best_model_name]

print("\nBest Model:", best_model_name)


# ==========================================
# 7. Train Best Model on Full Training Data
# ==========================================

best_model.fit(X_train, y_train)


# ==========================================
# 8. Evaluate on Test Set
# ==========================================

y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


# ==========================================
# 9. Display Results as DataFrame
# ==========================================

results_df = pd.DataFrame({
    "Model": cv_results.keys(),
    "CV Mean Accuracy": cv_results.values()
}).sort_values(by="CV Mean Accuracy", ascending=False)

print("\nModel Comparison Table:\n")
print(results_df)


Cross Validation Results (5-Fold):

Logistic Regression: 0.9802
Decision Tree: 0.9121
Random Forest: 0.9582
SVM: 0.9714
KNN: 0.9670

Best Model: Logistic Regression

Test Accuracy: 0.9824561403508771

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Confusion Matrix:

[[41  1]
 [ 1 71]]

Model Comparison Table:

                 Model  CV Mean Accuracy
0  Logistic Regression          0.980220
3                  SVM          0.971429
4                  KNN          0.967033
2        Random Forest          0.958242
1        Decision Tree          0.912088
